# Milestone 1: Data Profiling

Profil awal dataset CommonLingua-Train menggunakan Polars LazyFrame dan DuckDB. Dataset lokal terdiri dari dua file CSV (`commonlingua_train.csv` dan `commonlingua_validation.csv`) dengan kolom seperti `text`, `lang`, `source`, dan metadata lainnya. Notebook ini membaca file mentah tanpa mengubahnya.

Jalankan notebook dari folder proyek atau folder `notebooks`. Proses profiling membaca isi file; karena dataset berukuran besar, proses agregasi dapat memerlukan waktu.

In [26]:
%pip install duckdb polars

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [27]:
from pathlib import Path

import duckdb
import polars as pl

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
DATA_FILES = {
    'train': DATA_DIR / 'commonlingua_train.csv',
    'validation': DATA_DIR / 'commonlingua_validation.csv',
}

missing_files = [path for path in DATA_FILES.values() if not path.exists()]
if missing_files:
    missing_list = '\n'.join(str(path) for path in missing_files)
    raise FileNotFoundError(
        f'File dataset tidak ditemukan:\n{missing_list}\n'
        'Pastikan kedua CSV tersedia di data/raw/.'
    )

for split, path in DATA_FILES.items():
    size_mb = path.stat().st_size / 1_000_000
    print(f'{split}: {path.name} ({size_mb:,.1f} MB)')

train: commonlingua_train.csv (2,065.0 MB)
validation: commonlingua_validation.csv (226.5 MB)


## Profil Polars

`scan_csv` membangun query lazy; data dibaca saat `collect()` dipanggil. Profil berikut menghitung jumlah baris dan nilai null per kolom untuk masing-masing split.

In [28]:
def scan_dataset(path: Path) -> pl.LazyFrame:
    return pl.scan_csv(
        path,
        infer_schema_length=10_000,
        null_values=['', 'NULL', 'null'],
        try_parse_dates=False,
        low_memory=True,
    )


lazy_data = {split: scan_dataset(path) for split, path in DATA_FILES.items()}
profil_polars = {}

for split, frame in lazy_data.items():
    print(f'\nSkema {split}:')
    print(frame.collect_schema())

    ringkasan = frame.select(
        pl.len().alias('jumlah_baris'),
        pl.all().null_count().name.suffix('_null'),
    ).collect()
    profil_polars[split] = ringkasan
    print(f'Ringkasan {split}:')
    display(ringkasan)


Skema train:
Schema({'text': String, 'lang': String, 'source': String, 'identifier': String, 'title': String, 'collection': String, 'license': String, 'open_type': String, 'creator': String, 'date': String})
Ringkasan train:


jumlah_baris,text_null,lang_null,source_null,identifier_null,title_null,collection_null,license_null,open_type_null,creator_null,date_null
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
2482568,0,0,0,13836,47122,28989,26693,26693,2480877,56236



Skema validation:
Schema({'text': String, 'lang': String, 'source': String, 'identifier': String, 'title': String, 'collection': String, 'license': String, 'open_type': String, 'creator': String, 'date': String})
Ringkasan validation:


jumlah_baris,text_null,lang_null,source_null,identifier_null,title_null,collection_null,license_null,open_type_null,creator_null,date_null
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
272875,0,0,0,1558,5214,270,0,0,272694,3228


## Ringkasan Profiling

Jalankan sel kode berikut setelah profiling Polars untuk menampilkan ringkasan dataset.

In [29]:
jumlah_file = len(DATA_FILES)
ukuran_total_byte = sum(path.stat().st_size for path in DATA_FILES.values())
ukuran_total_gb = ukuran_total_byte / 1_000_000_000
jumlah_baris_per_split = {
    split: int(ringkasan['jumlah_baris'][0])
    for split, ringkasan in profil_polars.items()
}
jumlah_baris_total = sum(jumlah_baris_per_split.values())
jumlah_baris_format = f'{jumlah_baris_total:,}'.replace(',', '.')
ukuran_total_gb_format = f'{ukuran_total_gb:.2f}'.replace('.', ',')
jumlah_kolom_per_split = {
    split: len(frame.collect_schema().names())
    for split, frame in lazy_data.items()
}

if len(set(jumlah_kolom_per_split.values())) == 1:
    jumlah_kolom = str(next(iter(jumlah_kolom_per_split.values())))
else:
    jumlah_kolom = ', '.join(
        f'{split}: {jumlah}'
        for split, jumlah in jumlah_kolom_per_split.items()
    )

lebih_dari_satu_juta_baris = jumlah_baris_total > 1_000_000
minimal_lima_ratus_mb = ukuran_total_byte >= 500_000_000
memenuhi_syarat = lebih_dari_satu_juta_baris and minimal_lima_ratus_mb

print('=== RINGKASAN PROFILING ===')
print(f'Jumlah file        : {jumlah_file}')
print(f'Ukuran dataset     : {ukuran_total_gb_format} GB')
print(f'Jumlah baris       : {jumlah_baris_format}')
print(f'Jumlah kolom       : {jumlah_kolom}')
print(f'> 1 juta baris     : {"YA" if lebih_dari_satu_juta_baris else "TIDAK"}')
print(f'>= 500 MB          : {"YA" if minimal_lima_ratus_mb else "TIDAK"}')
print(f'Memenuhi syarat    : {"YA" if memenuhi_syarat else "TIDAK"}')

=== RINGKASAN PROFILING ===
Jumlah file        : 2
Ukuran dataset     : 2,29 GB
Jumlah baris       : 2.755.443
Jumlah kolom       : 10
> 1 juta baris     : YA
>= 500 MB          : YA
Memenuhi syarat    : YA


### Kesimpulan

Berdasarkan hasil profiling, dataset CommonLingua terdiri atas file train dan validation. Jumlah baris, ukuran total, serta jumlah kolom ditampilkan pada ringkasan di atas. Dataset memenuhi kriteria apabila jumlah barisnya lebih dari 1 juta dan ukuran totalnya minimal 500 MB.

### Contoh data

Ambil beberapa baris pertama dari setiap split untuk melihat bentuk rekaman tanpa mengumpulkan seluruh dataset.

In [30]:
for split, frame in lazy_data.items():
    print(f'Contoh {split}:')
    display(frame.head(5).collect())

Contoh train:


text,lang,source,identifier,title,collection,license,open_type,creator,date
str,str,str,str,str,str,str,str,str,str
"""55°51′16.63″ š. pl. 22°23′53.2…","""lit""","""Wikipedia""","""https://lt.wikipedia.org/wiki/…","""Žąsūgalos kalnas""","""Wikipedia""","""CC-BY-SA""","""Web""",null,"""2026"""
"""Thành phố Thiên Tân - Bột Hải…","""vie""","""Wikipedia""","""https://vi.wikipedia.org/wiki/…","""Danh sách báo chí Trung Quốc""","""Wikipedia""","""CC-BY-SA""","""Web""",null,"""2026"""
"""Ou konsidere mechan ki sou lat…","""hat""","""eBible""",null,null,"""eBible""","""Public Domain""","""Open_Culture""",null,"""1985"""
"""Vzreja Takahē z Južnega otoka…","""slv""","""Wikipedia""","""https://sl.wikipedia.org/wiki/…","""Takahe""","""Wikipedia""","""CC-BY-SA""","""Web""",null,"""2026"""
"""Loïens éstérnes - Zen-On Cont…","""pcd""","""Wikipedia""","""https://pcd.wikipedia.org/wiki…","""Akira Nishimura""","""Wikipedia""","""CC-BY-SA""","""Web""",null,"""2026"""


Contoh validation:


text,lang,source,identifier,title,collection,license,open_type,creator,date
str,str,str,str,str,str,str,str,str,str
"""Venera (simvol: ) — Həši siste…","""tly""","""Wikipedia""","""https://tly.wikipedia.org/wiki…","""Venera""","""Wikipedia""","""CC-BY-SA""","""Web""",null,"""2026"""
"""Lapkristė 14 dėina īr 318 metu…","""sgs""","""Wikipedia""","""https://bat-smg.wikipedia.org/…","""Lapkristė 14""","""Wikipedia""","""CC-BY-SA""","""Web""",null,"""2026"""
"""Gweler hefyd Cyhoeddodd Danie…","""cym""","""Wikipedia""","""https://cy.wikipedia.org/wiki/…","""Amore Che Vieni, Amore Che Vai""","""Wikipedia""","""CC-BY-SA""","""Web""",null,"""2026"""
"""Comestibilità A venta mai man…","""pms""","""Wikipedia""","""https://pms.wikipedia.org/wiki…","""Dendrocorticium ionides""","""Wikipedia""","""CC-BY-SA""","""Web""",null,"""2026"""
"""Самостоятелни романи - The Po…","""bul""","""Wikipedia""","""https://bg.wikipedia.org/wiki/…","""Кен Фолет""","""Wikipedia""","""CC-BY-SA""","""Web""",null,"""2026"""


## Profil DuckDB

DuckDB membaca CSV langsung dari disk dan menampilkan statistik ringkas per kolom. `SUMMARIZE` memindai data sehingga eksekusinya dapat memerlukan waktu. `strict_mode=false` membuat DuckDB tetap membaca CSV ketika menemukan baris dengan format tidak konsisten; periksa data sumber jika nilai pada baris bermasalah perlu dipastikan.

In [31]:
connection = duckdb.connect()
profil_duckdb = {}

for split, path in DATA_FILES.items():
    relation = connection.sql(
        'SUMMARIZE SELECT * FROM read_csv_auto(?, strict_mode=false, parallel=false)',
        params=[str(path)],
    )
    profil_duckdb[split] = pl.DataFrame(
        relation.fetchall(),
        schema=relation.columns,
        orient='row',
    )
    print(f'Profil DuckDB {split}:')
    display(profil_duckdb[split])

Profil DuckDB train:


column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,null,null,null,null,null,i64,"decimal[38,2]"
"""text""","""VARCHAR""","""…","""𱎼家话，又叫重安江苗语，是汉藏语系苗瑶语族一種闲话。 张济…",2323149,null,null,null,null,null,2482568,0.00
"""lang""","""VARCHAR""","""abk""","""zul""",350,null,null,null,null,null,2482568,0.00
"""source""","""VARCHAR""","""Anno""","""usc""",117,null,null,null,null,null,2482568,0.00
"""identifier""","""VARCHAR""","""-0xVHB9387w_1""","""（××××）××执异字第××号""",1901901,null,null,null,null,null,2482568,0.56
"""title""","""VARCHAR""",""" DARDUMAR VOA: Yadda Matashiya…","""󠀦󠀢󠀤󠀙󠀜󠀚󠀛-󠀦󠀢󠀤󠀙󠀜󠀚󠀛""",1574942,null,null,null,null,null,2482568,1.90
"""collection""","""VARCHAR""","""Achaemenid Royal Inscriptions …","""usc""",246,null,null,null,null,null,2482568,1.17
"""license""","""VARCHAR""","""CC-BY-SA""","""Various open science""",14,null,null,null,null,null,2482568,1.08
"""open_type""","""VARCHAR""","""Open_Culture""","""Web""",2,null,null,null,null,null,2482568,1.08
"""creator""","""VARCHAR""","""Barat et al.""","""Masoretic""",2,null,null,null,null,null,2482568,99.93


Profil DuckDB validation:


column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,null,null,null,null,null,i64,"decimal[38,2]"
"""text""","""VARCHAR""","""  Abstract: T…","""𧒽崗站（粵拼：leoi1 gong1 zaam6，英文：Le…",277830,null,null,null,null,null,272875,0.00
"""lang""","""VARCHAR""","""abk""","""zul""",337,null,null,null,null,null,272875,0.00
"""source""","""VARCHAR""","""Anno""","""reg_docs""",89,null,null,null,null,null,272875,0.00
"""identifier""","""VARCHAR""","""0000213163_34""","""（2021）黔0502民初12481号""",254827,null,null,null,null,null,272875,0.57
"""title""","""VARCHAR""",""" Gaza, sigidadenw tina se ka s…","""𧒽崗站 (地鐵)""",234288,null,null,null,null,null,272875,1.91
"""collection""","""VARCHAR""","""Achaemenid Royal Inscriptions …","""riao/The Royal Inscriptions of…",181,null,null,null,null,null,272875,0.10
"""license""","""VARCHAR""","""CC-BY-SA""","""Various open science""",11,null,null,null,null,null,272875,0.00
"""open_type""","""VARCHAR""","""Open_Culture""","""Web""",2,null,null,null,null,null,272875,0.00
"""creator""","""VARCHAR""","""Barat et al.""","""Masoretic""",2,null,null,null,null,null,272875,99.93


## Catatan Milestone 1

Isi temuan berdasarkan output profiling di atas:

- **Ukuran data:** catat ukuran train dan validation yang ditampilkan pada bagian awal notebook.
- **Jumlah baris:** catat `jumlah_baris` untuk masing-masing split.
- **Tipe kolom:** catat skema Polars dan periksa apakah inferensi tipe DuckDB berbeda.
- **Null terbanyak:** urutkan nilai `*_null` pada ringkasan Polars; sebutkan nama kolom dan jumlahnya.
- **Anomali untuk Milestone 2:** periksa rekaman kosong/duplikat, teks atau label bahasa yang tidak valid, serta perbedaan skema train dan validation. Jangan mengasumsikan anomali sebelum memeriksa hasil.